<a href="https://colab.research.google.com/github/beyzadurdu6619/TrustLLM-Uncertainty-Quantification/blob/main/notebooks/03_week/huggingface_llm_logits.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# [EN] Day 1: Installing Transformers and Understanding Tokenization
# [TR] 1. Gün: Transformers Kütüphanesi ve Tokenizer Mantığı

import warnings
warnings.filterwarnings("ignore") # Gereksiz uyarı mesajlarını gizler

from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B"

# Tokenizer'ı indiriyoruz
tokenizer = AutoTokenizer.from_pretrained(model_name)

prompt = "Language models are fascinating because"

# Encoding (Cümle -> Sayı)
input_ids = tokenizer.encode(prompt, return_tensors="pt")

print("[TR] Cümlenin Sayısal Kod Karşılıkları (Tokens):")
print(input_ids)

# Decoding (Sayı -> Cümle)
decoded_text = tokenizer.decode(input_ids[0])
print("\n[TR] Sayıların Tekrar Kelimelere Dönüşmüş Hali:")
print(decoded_text)

[TR] Cümlenin Sayısal Kod Karşılıkları (Tokens):
tensor([[13806,  4119,   525, 26291,  1576]])

[TR] Sayıların Tekrar Kelimelere Dönüşmüş Hali:
Language models are fascinating because


In [9]:
# [EN] Day 2: Loading an Open-Source LLM onto GPU
# [TR] 2. Gün: Açık Kaynaklı Bir LLM'i GPU'ya Yükleme

import torch
import warnings
warnings.filterwarnings("ignore")

from transformers import AutoModelForCausalLM

# torch_dtype yerine yeni standart olan 'dtype' kullanıyoruz
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,  # 'torch_dtype' uyarısını düzelten güncel kullanım
    device_map="auto"
)

print("[TR] Model Başarıyla GPU Hafızasına Yüklendi!")
print(f"[TR] Yüklenen Model Tipi: {type(model)}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[TR] Model Başarıyla GPU Hafızasına Yüklendi!
[TR] Yüklenen Model Tipi: <class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>


In [4]:
# [EN] Day 3 & 4: Forward Pass & Extracting Output Logits
# [TR] 3. ve 4. Gün: Modeli Çalıştırma ve Logitleri Yakalama

prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Modeli eğitmeyeceğimiz için hafızayı yormuyoruz (torch.no_grad)
with torch.no_grad():
    outputs = model(**inputs)

# outputs.logits -> [Batch_Size, Sequence_Length, Vocabulary_Size]
# Modelin ürettiği tüm kelimelerin ham skor matrisi!
logits = outputs.logits

print("[TR] Logit Matrisinin Boyutu:", logits.shape)
# Örn: [1, 5, 151936] -> 1 cümle, 5 kelime, sözlükteki 151.936 kelimenin her biri için skor!

[TR] Logit Matrisinin Boyutu: torch.Size([1, 5, 151936])


In [7]:
# [EN] Day 5 & 6: Converting Logits to Probabilities & Finding Top-5 Predictions
# [TR] 5. ve 6. Gün: Logitleri Olasılığa Çevirme ve En Yüksek 5 Tahmini Bulma

import torch.nn.functional as F

# Son kelimenin logitlerini alıyoruz
last_token_logits = logits[0, -1, :]

# 1. Haftadaki Softmax! (Skorları % olasılığa çevirir)
probabilities = F.softmax(last_token_logits, dim=-1)

# En yüksek olasılığa sahip ilk 5 kelimeyi alalım (Top-5)
top_5_probs, top_5_indices = torch.topk(probabilities, k=5)

print(f"[TR] Girdi Cümlesi: '{prompt}'")
print("[TR] Modelin Sıradaki Kelime İçin En Yüksek 5 Tahmini:")
print("-" * 50)

for prob, idx in zip(top_5_probs, top_5_indices):
    word = tokenizer.decode(idx.item())
    percentage = prob.item() * 100
    print(f"Kelime: '{word.strip()}' -> Olasılık: %{percentage:.2f}")

[TR] Girdi Cümlesi: 'The capital of France is'
[TR] Modelin Sıradaki Kelime İçin En Yüksek 5 Tahmini:
--------------------------------------------------
Kelime: 'Paris' -> Olasılık: %31.37
Kelime: '______' -> Olasılık: %11.54
Kelime: '____' -> Olasılık: %6.48
Kelime: '__' -> Olasılık: %5.54
Kelime: ':' -> Olasılık: %5.12
